# IPL Analytics (2008-2024)

<p align="center">
  <img src="https://dutchuncles.in/wp-content/uploads/2021/04/The-Biggest-Sporting-Tournament-of-the-Year.jpg" alt="Indian Premier League Logo" width="50%">
</p>


In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [53]:
df = pd.read_csv('matches.csv')
df.shape

(1095, 20)

In [54]:
df.head()

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


## Basic Data Cleaning

All the columns in the dataset are crucial for the purpose of analysis except the 'id' column. Thus, I'll be dropping it from the dataset

In [55]:
del df['id']

Backing all the other columns!

## Handling Missing Values

Firstly, we'll check how many null values are present in different columns of the dataset

In [56]:
df.isnull().sum()

season                0
city                 51
date                  0
match_type            0
player_of_match       5
venue                 0
team1                 0
team2                 0
toss_winner           0
toss_decision         0
winner                5
result                0
result_margin        19
target_runs           3
target_overs          3
super_over            0
method             1074
umpire1               0
umpire2               0
dtype: int64

### Resolving null values in the 'city' column

The 'city' column has 51 null values. However, the 'venue' column has no null values. From the 'venue' column, the names of the respecitve cities can be tracked. So, I've made a dictionary 'venue_to_city' which contains venue and their respecitve cities as key-value pairs. This will help in replacing the null values present in the 'city' column.

In [57]:
unique_pairs = df[['venue', 'city']].drop_duplicates()
venue_to_city = dict(zip(unique_pairs['venue'], unique_pairs['city']))
venue_to_city

{'M Chinnaswamy Stadium': 'Bangalore',
 'Punjab Cricket Association Stadium, Mohali': 'Chandigarh',
 'Feroz Shah Kotla': 'Delhi',
 'Wankhede Stadium': 'Mumbai',
 'Eden Gardens': 'Kolkata',
 'Sawai Mansingh Stadium': 'Jaipur',
 'Rajiv Gandhi International Stadium, Uppal': 'Hyderabad',
 'MA Chidambaram Stadium, Chepauk': 'Chennai',
 'Dr DY Patil Sports Academy': 'Mumbai',
 'Newlands': 'Cape Town',
 "St George's Park": 'Port Elizabeth',
 'Kingsmead': 'Durban',
 'SuperSport Park': 'Centurion',
 'Buffalo Park': 'East London',
 'New Wanderers Stadium': 'Johannesburg',
 'De Beers Diamond Oval': 'Kimberley',
 'OUTsurance Oval': 'Bloemfontein',
 'Brabourne Stadium': 'Mumbai',
 'Sardar Patel Stadium, Motera': 'Ahmedabad',
 'Barabati Stadium': 'Cuttack',
 'Brabourne Stadium, Mumbai': 'Mumbai',
 'Vidarbha Cricket Association Stadium, Jamtha': 'Nagpur',
 'Himachal Pradesh Cricket Association Stadium': 'Dharamsala',
 'Nehru Stadium': 'Kochi',
 'Holkar Cricket Stadium': 'Indore',
 'Dr. Y.S. Rajasekha

Filling in the null values present in the 'city' column based upon the mapping dictionary created above. 

In [58]:
df['city'] = df['city'].fillna(df['venue'].map(venue_to_city)) 

### Resolving null values in the 'result_margin' column

In [59]:
null_rows = df[df['result_margin'].isnull()] # filtering rows where result_margin is null
null_result_values = null_rows['result'].unique() # checking the corresponding values for 'null_rows' in result column
print(null_result_values)


['tie' 'no result']


From the above result, it can be clearly stated that all the null values in the 'result_margin' column arise due to the result being a 'tie' or 'no result'. In such cases, the 'result_margin' will be 0. Thus, replacing all the null values with a 0. 

In [60]:
df['result_margin'].fillna(0)

0       140.0
1        33.0
2         9.0
3         5.0
4         5.0
        ...  
1090      4.0
1091      8.0
1092      4.0
1093     36.0
1094      8.0
Name: result_margin, Length: 1095, dtype: float64

### Resolving null values in the 'winner' column

In [61]:
null_rows = df[df['winner'].isnull()] # filtering rows where winner values are null
null_result_values = null_rows['result'].unique() # checking the corresponding values for 'result' in result column
print(null_result_values)


['no result']


It is thereby clear from the above outcome that null values in the 'winner' column arise in the instances 'no result'. Hence, we can replace these null values with 'no winner'

In [62]:
df['winner'].fillna('no winner', inplace=True)

C:\Users\Dell\AppData\Local\Temp\ipykernel_2804\5659939.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['winner'].fillna('no winner', inplace=True)


### Resolving null values in the 'player_of_match' column

In [63]:
null_rows = df[df['player_of_match'].isnull()] # filtering rows where winner values are null
null_result_values = null_rows['result'].unique() # checking the corresponding values for 'result' in result column
print(null_result_values)

['no result']


It is thereby clear from the above outcome that null values in the 'player_of_match' column arise in the instances 'no result'. Hence, we can replace these null values with 'no awardee'

In [64]:
df['player_of_match'].fillna('no awardee', inplace=True)

C:\Users\Dell\AppData\Local\Temp\ipykernel_2804\4188835761.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['player_of_match'].fillna('no awardee', inplace=True)


### Resolving null values in the 'target_runs' and 'target_overs' column

In [65]:
null_rows = df[df['target_runs'].isnull()] # filtering rows where winner values are null
null_result_values = null_rows['result'].unique() # checking the corresponding values for 'result' in result column
print(null_result_values)

['no result']


In [66]:
null_rows = df[df['target_overs'].isnull()] # filtering rows where winner values are null
null_result_values = null_rows['result'].unique() # checking the corresponding values for 'result' in result column
print(null_result_values)

['no result']


In this case too, the reason is same i.e. 'no result'. Thus changing null values in both columns with 'match abandoned'

In [67]:
df['target_runs'].fillna(0)
df['target_overs'].fillna('match abandoned')

0       20.0
1       20.0
2       20.0
3       20.0
4       20.0
        ... 
1090    20.0
1091    20.0
1092    20.0
1093    20.0
1094    20.0
Name: target_overs, Length: 1095, dtype: object

### Resolving null values in the 'method' column

1074 columns having null values means that no special method like the DLS (Duckworth-Lewis-Stern) method was used in those particular matches. This means that these matches were held under normal circumstances without any climatic hinderance. Thus, I've replaced all these null values with 'normal'

In [68]:
df['method'].fillna('normal', inplace=True)

C:\Users\Dell\AppData\Local\Temp\ipykernel_2804\2938777472.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['method'].fillna('normal', inplace=True)


Checking if all the null values have been removed from the dataset or not:

In [69]:
df.isnull().sum()

season              0
city                0
date                0
match_type          0
player_of_match     0
venue               0
team1               0
team2               0
toss_winner         0
toss_decision       0
winner              0
result              0
result_margin      19
target_runs         3
target_overs        3
super_over          0
method              0
umpire1             0
umpire2             0
dtype: int64

Thus, all the null values have been handled successfully.

In [70]:
df.head()

,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,normal,Asad Rauf,RE Koertzen
1,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,normal,MR Benson,SL Shastri
2,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,normal,Aleem Dar,GA Pratapkumar
3,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,normal,SJ Davis,DJ Harper
4,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,normal,BF Bowden,K Hariharan


### Making the Values of the Season Column Consistent

In [92]:
season_replacement_dict = {'2007/08': '2008', '2009/10': '2010', '2020/21': '2020'}
df['season'] = df['season'].replace(season_replacement_dict)

In [106]:
df['season'].unique()

array(['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015',
       '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023',
       '2024'], dtype=object)

All values from the season column have been made continuous in the 'yyyy' format.

The data cleaning process was completed effortlessly. Have a look at the cleaned dataset: 

In [73]:
df.head()

,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,2008,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,normal,Asad Rauf,RE Koertzen
1,2008,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,normal,MR Benson,SL Shastri
2,2008,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,normal,Aleem Dar,GA Pratapkumar
3,2008,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,normal,SJ Davis,DJ Harper
4,2008,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,normal,BF Bowden,K Hariharan


In [74]:
df.to_csv('ipl_all_matches_data_cleaned.csv')

# Cleaning the Second Data Set

In [75]:
df1 = pd.read_csv('ipl_ball_by_ball_data.csv')
df1.shape

C:\Users\Dell\AppData\Local\Temp\ipykernel_2804\664166655.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('ipl_ball_by_ball_data.csv')


(255759, 19)

In [76]:
df1.head()

,Match id,Date,Season,Batting team,Bowling team,Innings No,Ball No,Bowler,Striker,Non Striker,runs_scored,extras,type of extras,score,score/wicket,wicket_confirmation,wicket_type,fielders_involved,Player Out
0,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.1,P Kumar,SC Ganguly,BB McCullum,0,1,legbyes,1,1/0,0,NaN,NaN,NaN
1,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.2,P Kumar,BB McCullum,SC Ganguly,0,0,NaN,1,1/0,0,NaN,NaN,NaN
2,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.2,P Kumar,BB McCullum,SC Ganguly,0,1,wides,2,2/0,0,NaN,NaN,NaN
3,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.3,P Kumar,BB McCullum,SC Ganguly,0,0,NaN,2,2/0,0,NaN,NaN,NaN
4,335982,2008-04-18,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.4,P Kumar,BB McCullum,SC Ganguly,0,0,NaN,2,2/0,0,NaN,NaN,NaN


In [77]:
df1.isnull().sum()

Match id                    0
Date                        0
Season                      0
Batting team                0
Bowling team                0
Innings No                  0
Ball No                     0
Bowler                      0
Striker                     0
Non Striker                 0
runs_scored                 0
extras                      0
type of extras         241936
score                       0
score/wicket                0
wicket_confirmation         0
wicket_type            243108
fielders_involved      246637
Player Out             243108
dtype: int64

In [78]:
df1.columns

Index(['Match id', 'Date', 'Season', 'Batting team', 'Bowling team',
       'Innings No', 'Ball No', 'Bowler', 'Striker', 'Non Striker',
       'runs_scored', 'extras', 'type of extras', 'score', 'score/wicket',
       'wicket_confirmation', 'wicket_type', 'fielders_involved',
       'Player Out'],
      dtype='object')

In [79]:
df1[['type of extras', 'wicket_type', 'fielders_involved','Player Out']] = df1[['type of extras', 'wicket_type', 'fielders_involved','Player Out']].fillna('none')

In [80]:
df1.isnull().sum()

Match id               0
Date                   0
Season                 0
Batting team           0
Bowling team           0
Innings No             0
Ball No                0
Bowler                 0
Striker                0
Non Striker            0
runs_scored            0
extras                 0
type of extras         0
score                  0
score/wicket           0
wicket_confirmation    0
wicket_type            0
fielders_involved      0
Player Out             0
dtype: int64

In [ ]:
df1 = df1.drop(columns=['Match id', 'Date'])

In [87]:
df1 = df1.drop(columns=['Date'])

In [82]:
df1.isnull().sum()

Date                   0
Season                 0
Batting team           0
Bowling team           0
Innings No             0
Ball No                0
Bowler                 0
Striker                0
Non Striker            0
runs_scored            0
extras                 0
type of extras         0
score                  0
score/wicket           0
wicket_confirmation    0
wicket_type            0
fielders_involved      0
Player Out             0
dtype: int64

In [88]:
df1.head()

,Season,Batting team,Bowling team,Innings No,Ball No,Bowler,Striker,Non Striker,runs_scored,extras,type of extras,score,score/wicket,wicket_confirmation,wicket_type,fielders_involved,Player Out
0,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.1,P Kumar,SC Ganguly,BB McCullum,0,1,legbyes,1,1/0,0,none,none,none
1,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.2,P Kumar,BB McCullum,SC Ganguly,0,0,none,1,1/0,0,none,none,none
2,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.2,P Kumar,BB McCullum,SC Ganguly,0,1,wides,2,2/0,0,none,none,none
3,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.3,P Kumar,BB McCullum,SC Ganguly,0,0,none,2,2/0,0,none,none,none
4,2007/08,Kolkata Knight Riders,Royal Challengers Bangalore,1,0.4,P Kumar,BB McCullum,SC Ganguly,0,0,none,2,2/0,0,none,none,none


In [103]:
df1['Season'].unique()

array(['2007/08', '2009', '2009/10', '2011', '2012', 2012, 2013, 2014,
       2015, 2016, 2017, 2018, '2018', '2019', '2020', '2021', 2021, 2022,
       2023, 2024], dtype=object)

The following function cleans the 'Season' column:
1. Extracts the year after the slash (e.g., '2007/08' -> '2008').
2. Removes single quotes (e.g., "'2009'" -> '2009').
3. Keeps integers unchanged.


In [ ]:
def format_season(value):
    if isinstance(value, str):  
        if '/' in value:  
            parts = value.split('/')  
            if len(parts[1]) == 2:  
                return "20" + parts[1]  
            return parts[1]  
        else:
            return value.strip("'")  
    return value  

df1['Season'] = df1['Season'].apply(format_season)

df1['Season'] = df1['Season'].astype(int)

In [112]:
df1['Season'].unique()

array([2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018,
       2019, 2020, 2021, 2022, 2023, 2024])

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255759 entries, 0 to 255758
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Date                 255759 non-null  object 
 1   Season               255759 non-null  object 
 2   Batting team         255759 non-null  object 
 3   Bowling team         255759 non-null  object 
 4   Innings No           255759 non-null  int64  
 5   Ball No              255759 non-null  float64
 6   Bowler               255759 non-null  object 
 7   Striker              255759 non-null  object 
 8   Non Striker          255759 non-null  object 
 9   runs_scored          255759 non-null  int64  
 10  extras               255759 non-null  int64  
 11  type of extras       255759 non-null  object 
 12  score                255759 non-null  int64  
 13  score/wicket         255759 non-null  object 
 14  wicket_confirmation  255759 non-null  int64  
 15  wicket_type      

In [114]:
df1.to_csv("ipl_ball_by_ball_data_cleaned.csv")

This dataset seems pretty clean now. We have, 2 datasets ready for our visualization. 

The visualization part will be carried out in Microsoft Power BI simply because of the abundant visualization options it offers. See you there!

In [ ]:
df.columns

Index(['season', 'city', 'date', 'match_type', 'player_of_match', 'venue',
       'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result',
       'result_margin', 'target_runs', 'target_overs', 'super_over', 'method',
       'umpire1', 'umpire2'],
      dtype='object')

In [ ]:
df1.columns

Index(['Date', 'Season', 'Batting team', 'Bowling team', 'Innings No',
       'Ball No', 'Bowler', 'Striker', 'Non Striker', 'runs_scored', 'extras',
       'type of extras', 'score', 'score/wicket', 'wicket_confirmation',
       'wicket_type', 'fielders_involved', 'Player Out'],
      dtype='object')

In [ ]:
df1.isnull().sum()

Date                   0
Season                 0
Batting team           0
Bowling team           0
Innings No             0
Ball No                0
Bowler                 0
Striker                0
Non Striker            0
runs_scored            0
extras                 0
type of extras         0
score                  0
score/wicket           0
wicket_confirmation    0
wicket_type            0
fielders_involved      0
Player Out             0
dtype: int64

In [ ]:
df1['wicket_type'].unique()

array(['none', 'caught', 'bowled', 'run out', 'lbw', 'retired hurt',
       'stumped', 'caught and bowled', 'hit wicket',
       'obstructing the field', 'retired out'], dtype=object)

See I want to make 3 pages:

1st page, overall IPL stats

2nd page, team stats. 
Overview: give an option to select a team from all the teams present. then following statistics need to be shown: total matches played by the team, number of matches won, number of matches lost, number of matches tie, number of matches drawn, number of matches no result, win percentage, number of matches toss won, number of matches decided to bat first, number of matches decided to bowl first.

3rd page, player stats.
Overview: give an option to first select a team and then all its player would be visible. select a player from that team. upon selecting a player their batting stats should be visible by default and at the same time an option to switch to bowling stats, and fielding stats should also be there. 

In batting stats include the following: player image, total matches, total innings, not out, total runs, highest score, batting avg, strike rate, 30s, 50s, 100s, 4s, 6s, ducks, matches won, matches lost.

In bowling stats include the following: player image, matches, innings, overs, maiden overs, runs given , wickets clinched totoal, best bowling figures, 3 wicket haul, 5 wicket haul, no of hattricks, bowling economy, strike rate bowling, bowling avg, total wides bowled, total no balls bowled, total dots bowled, 4s conceeded, 6s conceeded.

In fielding stats include the following: player image, total matches, total catches, catch and bowled, run outs, stumpings.  